In [29]:

import torch
import torch.nn as nn
# import sys 
# sys.path.append(".")
import nbimporter
from GPT_Architecture import GPTModel
GPT_CONFIG_124M={
 "vocab_size":50257,
 "context_length":256,
 "emb_dim":768,
 "n_heads":12,
 "n_layers":12,
 "drop_rate":0.1,
 "qkv_bias":False
}

In [30]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Where there is"
txt2 = "there is a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)


tensor([[8496,  612,  318],
        [8117,  318,  257]])


In [31]:
torch.manual_seed(45)
model = GPTModel(GPT_CONFIG_124M)
logits = model(batch)
print("Output Shape",logits.shape)
print(logits)

Output Shape torch.Size([2, 3, 50257])
tensor([[[ 0.2904,  0.2883, -0.1145,  ...,  0.3583, -0.2279,  0.1270],
         [ 0.1474,  0.2985, -0.7177,  ...,  1.0781, -0.3418, -0.0047],
         [ 0.2735,  0.3138, -0.1868,  ..., -0.1459, -0.5435, -0.6475]],

        [[-0.1339, -0.3756,  0.3700,  ..., -0.8877,  0.6405, -0.0065],
         [ 0.6721,  1.1726,  0.2171,  ...,  0.1202,  0.3641, -0.7155],
         [ 0.0990, -0.9672, -0.2140,  ...,  0.8467,  0.6022,  0.3638]]],
       grad_fn=<UnsafeViewBackward0>)


In [32]:
inputs = torch.tensor([[16833, 3626, 6100], ##["every effort moves"]
                       [40,1107,588]]) ##["I realy like"]
targets = torch.tensor([[3626, 6100,345], ##["effort moves you"]
                       [1107,588,11311]]) ##["really like chocolate"]


In [33]:
with torch.no_grad():
    logits = model(inputs)
probas = torch.softmax(logits, dim=-1) #Probability of each token in vocabulary
print(probas.shape) #Shape: (batch_size, num_tokens, vocab_size)


torch.Size([2, 3, 50257])


In [34]:
token_id = torch.argmax(probas, dim=-1, keepdim=True)
print(f"Token IDs: {token_id}")

Token IDs: tensor([[[45716],
         [37342],
         [ 9113]],

        [[31020],
         [49390],
         [23732]]])


In [37]:
text_idx =0
target_probas_1 = probas[text_idx,[0,1,2], targets[text_idx]]
print(f"Text 1: {target_probas_1}")

text_idx = 1
target_probas_2 = probas[text_idx,[0,1,2], targets[text_idx]]
print(f"Text 2: {target_probas_2}")


Text 1: tensor([2.2204e-05, 1.4445e-05, 6.8143e-05])
Text 2: tensor([9.6963e-06, 2.8625e-05, 1.6781e-05])


In [38]:
log_probas = torch.log(torch.cat((target_probas_1,target_probas_2)))
print(log_probas)

tensor([-10.7152, -11.1452,  -9.5939, -11.5438, -10.4612, -10.9953])


In [39]:
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

tensor(-10.7424)


In [41]:
neg_log_probas = avg_log_probas * -1 #This is used for minimizing the loss without negative the loss will increase
##Negative wont decrease without -ve log
print(neg_log_probas)

tensor(10.7424)
